In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler,LabelEncoder

In [ ]:
df = pd.read_csv("../data/lifestyle_disease_transformed.csv")
df = df.drop(["Unnamed: 0"], axis=1)

target = df.iloc[:,1].values
features = df.iloc[:,2:-1].values

le = LabelEncoder()
ss = StandardScaler()

target = le.fit_transform(target).reshape(-1,1)
features = ss.fit_transform(features)

target = df.iloc[:,-1:].values
features = df.iloc[:,:-1].values

x_train = features[:1000]
y_train = target[:1000]
x_test = features[1000:1500]
y_test = target[1000:1500]

In [ ]:
import heapq

class Classifier:
    def __init__(self, n_neighbors=3):
        self.n_neighbors = n_neighbors
    
    def fit(self, x_train, y_train):
        self.x_train = x_train
        self.y_train = y_train
    
    def distance(self, x, y):
        return np.sum((x - y) ** 2)
    
    def predict_single(self, x):
        heap = []
        
        for i in range(len(self.x_train)):
            dist = self.distance(self.x_train[i], x)

            heapq.heappush(heap, (-dist, self.y_train[i]))

            if len(heap) > self.n_neighbors:
                heapq.heappop(heap) 
        
        labels = [int(label) for (_, label) in heap]
        
        count = np.bincount(labels)
        return np.argmax(count)
    
    def predict(self, x_test):
        return np.array([self.predict_single(x) for x in x_test]).reshape(-1, 1)


In [ ]:
model = Classifier(n_neighbors=3)
model.fit(x_train,y_train)
y_pred = model.predict(x_test)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,confusion_matrix,auc,roc_auc_score,roc_curve
def statistical_outcomes(x,y):
    mat = confusion_matrix(x,y)
    acc = accuracy_score(x,y)
    pre = precision_score(x,y)
    rec = recall_score(x,y)
    f1s = f1_score(x,y)
    _roc = roc_auc_score(x,y)
    print("Confusion matrix: \n",mat)
    print("Accuracy Score: \n",acc)
    print("Precision Score: \n",pre)
    print("Recall Score: \n",rec)
    print("F1 Score: \n",f1s)
    print("ROC_AUC Score: \n",_roc)
    p,q,r = roc_curve(y_test,y_pred)
    plt.plot(p,q)
    plt.xlabel("fpr")
    plt.ylabel("tpr")
    plt.title("Roc_curve")
    plt.show()


In [ ]:
statistical_outcomes(y_test,y_pred)